# EventHallusion + VideoLLaVA on Colab

Notebook này chạy benchmark EventHallusion theo đúng mục tiêu Statement 4:

- lấy khoảng 200 video test
- kiểm chứng language prior và context bias với `misleading`
- kiểm chứng rare event xuyên suốt video với `entire`
- kiểm chứng common-rare mix với `mix`
- so sánh `normal` và `spatial_gaussian`

Output sẽ được lưu ra CSV/JSON trong thư mục kết quả để bạn đem đi phân tích.

In [1]:
# Install dependencies
!pip -q install transformers accelerate bitsandbytes decord av huggingface_hub tqdm pandas opencv-python
!pip -q install git+https://github.com/facebookresearch/pytorchvideo.git@28fe037d212663c6a24f373b94cc5d478c8c1a1d


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 13.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# Clone the benchmark repo
import os
repo_dir = "/content/EventHallusion"
if os.path.exists(repo_dir):
    !git -C /content/EventHallusion pull --rebase origin master
else:
    !git clone https://github.com/NguyenDucThang-tb/EventHallusion.git /content/EventHallusion
%cd /content/EventHallusion


Cloning into '/content/EventHallusion'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 142 (delta 13), reused 18 (delta 7), pack-reused 117 (from 1)
Receiving objects: 100% (142/142), 87.64 MiB | 28.62 MiB/s, done.
Resolving deltas: 100% (29/29), done.
Updating files: 100% (93/93), done.
/content/EventHallusion


In [3]:
# Optional: mount Google Drive if you want to keep videos/results there
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Prepare data

You need the EventHallusion videos extracted to a folder. The files in `questions/` are only annotations / question sets, not videos.

The video folder is not included in the repo checkout. Download the EventHallusion video release from the Google Drive link in the README, then extract it to `/content/EventHallusion/videos` or to a Drive path you mount in Colab.

Typical paths:

- questions: `/content/EventHallusion/questions`
- videos: `/content/EventHallusion/videos`  # create this by extracting the downloaded video archive
- results: `/content/EventHallusion/results`

If your videos are on Drive, set `video_root` to that path instead.

In [4]:
# Load Video-LLaVA
import torch
from transformers import VideoLlavaForConditionalGeneration, VideoLlavaProcessor, BitsAndBytesConfig

model_name = "LanguageBind/Video-LLaVA-7B-hf"
processor = VideoLlavaProcessor.from_pretrained(model_name)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = VideoLlavaForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model loaded")


preprocessor_config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

[transformers] Requested torchvision backend is not available. Falling back to pil backend.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.59k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1077 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/148 [00:00<?, ?B/s]

Model loaded


In [5]:
!pip -q install gdown
!gdown --fuzzy "https://drive.google.com/file/d/1IPmx6Y80UrXwVPmZJh6zjCPHtlsw4p9n/view?usp=sharing" -O EventHallusion_videos.zip
!mkdir -p /content/EventHallusion/videos
!unzip -q EventHallusion_videos.zip -d /content/EventHallusion/videos

Downloading...
From (original): https://drive.google.com/uc?id=1IPmx6Y80UrXwVPmZJh6zjCPHtlsw4p9n
From (redirected): https://drive.google.com/uc?id=1IPmx6Y80UrXwVPmZJh6zjCPHtlsw4p9n&confirm=t&uuid=80b5cc88-b3a2-4b68-95e1-eff0ce0c98e5
To: /content/EventHallusion/EventHallusion_videos.zip
100% 4.98G/4.98G [01:09<00:00, 71.7MB/s]


In [6]:
# Reload the helper and validate the data paths before running the benchmark
from pathlib import Path
import importlib
import eventhallusion_videollava_eval

importlib.reload(eventhallusion_videollava_eval)
from eventhallusion_videollava_eval import compare_conditions

questions_root = Path('/content/EventHallusion/questions')
video_root = Path('/content/EventHallusion/videos')
print('questions exists:', questions_root.exists())
print('videos exists:', video_root.exists())
print('question files:', sorted([p.name for p in questions_root.glob('*.json')]))
videos = sorted(video_root.rglob('*.mp4'))
print('video count:', len(videos))
print('sample videos:', [p.name for p in videos[:5]])


questions exists: True
videos exists: True
question files: ['entire_questions.json', 'misleading_questions.json', 'mix_questions.json']
video count: 397
sample videos: ['entire_001.mp4', 'entire_002.mp4', 'entire_003.mp4', 'entire_004.mp4', 'entire_005.mp4']


In [7]:
# Run EventHallusion benchmark
questions_root = "/content/EventHallusion/questions"
video_root = "/content/EventHallusion/videos"
out_dir = "/content/EventHallusion/results"

# 200 videos total, split evenly across the three EventHallusion groups
summary = compare_conditions(
    model=model,
    processor=processor,
    questions_root=questions_root,
    video_root=video_root,
    out_dir=out_dir,
    n_total_videos=200,
    n_frames=4,
    sigma=25,
    seed=42,
    per_split={"misleading": 67, "entire": 67, "mix": 66},
)

summary


[normal] samples: 200


normal videos:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


normal:misleading_093:   0%|          | 0/2 [00:00<?, ?it/s]

normal:entire_090:   0%|          | 0/2 [00:00<?, ?it/s]

...

spatial_gaussian:entire_009:   0%|          | 0/2 [00:00<?, ?it/s]

spatial_gaussian:misleading_090:   0%|          | 0/2 [00:00<?, ?it/s]

spatial_gaussian:misleading_094:   0%|          | 0/2 [00:00<?, ?it/s]

Saved:
{'csv': '/content/EventHallusion/results/eventhallusion_normal.csv', 'json': '/content/EventHallusion/results/eventhallusion_normal.json'}
{'csv': '/content/EventHallusion/results/eventhallusion_spatial_gaussian.csv', 'json': '/content/EventHallusion/results/eventhallusion_spatial_gaussian.json'}
/content/EventHallusion/results/eventhallusion_summary.csv
       condition  overall  acc_entire  acc_misleading  acc_mix
          normal 0.411483    0.361111        0.464789 0.409091
spatial_gaussian 0.406699    0.361111        0.436620 0.424242


,condition,overall,acc_entire,acc_misleading,acc_mix
0,normal,0.411483,0.361111,0.464789,0.409091
1,spatial_gaussian,0.406699,0.361111,0.436620,0.424242


## What to look at

- `acc_misleading`: language prior / context bias
- `acc_entire`: whether the model follows the whole video timeline
- `acc_mix`: whether the model can avoid frame-level shortcut on mixed rare/common events

Compare `normal` vs `spatial_gaussian`.
If spatial Gaussian helps mostly on `misleading`, then it mainly attacks spatial/context shortcuts rather than temporal reasoning itself.

In [8]:
# Download results to Drive if needed
import shutil
drive_out = "/content/drive/MyDrive/EventHallusion_results"
os.makedirs(drive_out, exist_ok=True)
if os.path.exists(out_dir):
    for name in [
        "eventhallusion_summary.csv",
        "eventhallusion_normal.csv",
        "eventhallusion_spatial_gaussian.csv",
        "eventhallusion_normal_predictions.json",
        "eventhallusion_spatial_gaussian_predictions.json",
    ]:
        src = os.path.join(out_dir, name)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(drive_out, name))
print(drive_out)


/content/drive/MyDrive/EventHallusion_results
